# Notebook 10 — Your Workflow

**Premise:** The point of the previous five notebooks wasn't Arc.
It was the *thinking*. Build a harness around a model:

1. **Goal** — what is the agent for? One sentence.
2. **Knowledge** — what does it need to know? (system prompt + skills)
3. **Capabilities** — what does it need to *do*? (tools)
4. **Boundaries** — what must it *not* do? (sandbox, policy)
5. **Evidence** — how do we know it worked? (audit, verification)

If you can answer those five questions, you can build an agent — in
Arc, in LangChain, in raw Python, anywhere.

This notebook walks one example end-to-end, then leaves space for you
to build your own.

## Worked example: the Run Triage agent

**The task:** scientist hands the agent a run id. Agent reads the
log, classifies any anomalies against the known failure modes, and
writes a one-paragraph triage note recommending an action.

Walk through the five questions:

| Question | Answer |
|---|---|
| Goal | Triage an experiment run from its log |
| Knowledge | `log_analyst` skill (the `SKILL.md` in `skills/log_analyst/`, authored in the format from notebook 04) |
| Capabilities | `read_log(run_id)`, `grep_log(run_id, pattern)`, `skill_lookup(reference)`, `write_triage_note(text)` |
| Boundaries | Read-only on the log store; write only to the triage notes folder |
| Evidence | Capture all events, verify chain, save the triage note |

## Setup

**Imports first.** The pieces we need: `connect()` (the one-switch model helper) and the `run` loop plus its building blocks (`Tool`, `ToolContext`, `SandboxConfig`, `StaticProvider`).

In [ ]:
from pathlib import Path
from textwrap import dedent
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from roadshow import connect
from arcrun import run, Tool, ToolContext, SandboxConfig, StaticProvider
from rich import print

**Wire the model and the paths.** Load the model, then resolve the skill folder and the triage-notes output folder (creating it if needed).

In [ ]:
model = connect()
SKILL = Path('../skills/log_analyst').resolve()
TRIAGE_DIR = Path('../data/triage').resolve()
TRIAGE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Knowledge — load the skill

**Read the skill into a string.** This is the `log_analyst` skill —
the `SKILL.md` in `skills/log_analyst/`, written in the format you
learned in notebook 04. We print it so you can see exactly what the
agent will be told.

In [ ]:
skill_md = (SKILL / 'SKILL.md').read_text()
print(skill_md)

## 2. Capabilities — three tools

Each tool is one function. The bounds of what the agent can affect
are *exactly* the union of these three tools — nothing more.

We point `read_log` at **real log files** in `../data/logs/` —
around 12,000 lines combined. The agent has to skim them, find
the anomaly, and reason about scope. This is closer to the real
thing than a 5-line stub.

**Point at the real logs.** This is editable — add your own files to `data/logs/` and list them here. We print line counts and sizes so you can see how big they really are.

In [ ]:
LOG_DIR = Path('../data/logs').resolve()

# 🔧 EDIT ME — list the real logs available; add your own to data/logs/ to extend.
LOG_FILES = {
    '8841':       LOG_DIR / 'run_8841.log',
    'oss-04':     LOG_DIR / 'storage_oss-04_2026-04-22.log',
}

for rid, p in LOG_FILES.items():
    n = sum(1 for _ in p.open())
    print(f'  {rid:<8s} {p.name:<40s} {n:>6d} lines  {p.stat().st_size:>10,d} bytes')

**Tool 1 — `read_log`.** Fetch a log's full text by run id, but cap the bytes returned so one call can't flood the window. Big logs get truncated with a hint to use `grep_log`.

In [ ]:
async def read_log(args, ctx: ToolContext) -> str:
    """Read a real log file — but enforce a max byte budget so a single tool call
    never floods the agent's context. Real ops tools do this too."""
    rid = args['run_id']
    p = LOG_FILES.get(rid)
    if p is None or not p.exists():
        return f'No log for run {rid}. Available: {list(LOG_FILES)}'
    text = p.read_text()
    MAX_BYTES = 60_000   # ~ 1k lines, comfortably fits in context
    if len(text) > MAX_BYTES:
        return text[:MAX_BYTES] + f'\n\n[…{len(text)-MAX_BYTES} more bytes truncated. Use grep_log to search.]'
    return text

**Tool 2 — `grep_log`.** Search a log for a substring (case-insensitive) and return matching lines. This is how the agent skims a 12,000-line log without reading it all.

In [ ]:
async def grep_log(args, ctx: ToolContext) -> str:
    """Find lines in a log matching a substring (case-insensitive). Useful for
    huge logs the agent shouldn't read in full."""
    rid, pat = args['run_id'], args['pattern'].lower()
    p = LOG_FILES.get(rid)
    if p is None:
        return f'No log for run {rid}'
    hits = [ln for ln in p.read_text().splitlines() if pat in ln.lower()]
    return '\n'.join(hits[:80]) + (f'\n[showing first 80 of {len(hits)} hits]' if len(hits) > 80 else '')

**Tool 3 — `skill_lookup`.** Fetch a reference file from the skill folder, e.g. the known-failure-modes catalog.

In [ ]:
async def skill_lookup(args, ctx: ToolContext) -> str:
    ref = args['reference']
    p = SKILL / ref
    return p.read_text() if p.exists() else f'No reference: {ref}'

**Tool 4 — `write_triage_note`.** Save the agent's final paragraph to the triage-notes folder. This is the agent's *only* write capability.

In [ ]:
async def write_triage_note(args, ctx: ToolContext) -> str:
    rid = args['run_id']
    out = TRIAGE_DIR / f'run_{rid}_{datetime.now():%Y%m%d_%H%M%S}.md'
    out.write_text(args['note'])
    return f'Wrote {out}'

**Register the four tools.** Each `Tool` pairs a name and description with an input schema and the function above. This list *is* the full set of things the agent can do.

In [ ]:
tools = [
    Tool(name='read_log', description='Fetch the full text of a log by run id (truncates at 60KB).',
         input_schema={'type': 'object', 'properties': {'run_id': {'type': 'string', 'description': 'Run id, e.g. 8841 or oss-04'}}, 'required': ['run_id']},
         execute=read_log),
    Tool(name='grep_log', description='Search a log for a substring (case-insensitive); returns matching lines with context.',
         input_schema={'type': 'object', 'properties': {'run_id': {'type': 'string'}, 'pattern': {'type': 'string'}}, 'required': ['run_id','pattern']},
         execute=grep_log),
    Tool(name='skill_lookup', description='Fetch a reference file. Available: references/known_failure_modes.md',
         input_schema={'type': 'object', 'properties': {'reference': {'type': 'string'}}, 'required': ['reference']},
         execute=skill_lookup),
    Tool(name='write_triage_note', description='Save the final triage note for a run.',
         input_schema={'type': 'object', 'properties': {'run_id': {'type': 'string'}, 'note': {'type': 'string'}}, 'required': ['run_id', 'note']},
         execute=write_triage_note),
]

## 3. Boundaries — the sandbox

Right now the sandbox is permissive (all 3 tools allowed). In
production you'd allowlist by *role* — a triage agent shouldn't have
`delete_log` or `modify_run`. The `SandboxConfig.allowed_tools` field
is your enforcement point.

**Allowlist the tools.** Only tools named here can fire — the enforcement point for what the agent is permitted to touch.

In [ ]:
sandbox = SandboxConfig(
    allowed_tools=['read_log', 'grep_log', 'skill_lookup', 'write_triage_note'],
)

## 4. Run with full evidence capture

**Build the system prompt.** Set up the event log, then compose the agent's instructions — its role plus the skill procedure we loaded above.

In [ ]:
events = []

system_prompt = dedent(f'''
    You are an experiment Run Triage agent at a national lab.
    For any run id, fetch the log, identify anomalies using the skill below,
    and write a one-paragraph triage note recommending the next action.
    Always end by calling write_triage_note with your final paragraph.

    {skill_md}
''').strip()

**Pick which run to triage.** Editable: `'8841'` (HPC run with a DRAM ECC fault) or `'oss-04'` (storage incident).

In [ ]:
# 🔧 EDIT ME — try '8841' (HPC run with DRAM ECC fault) or 'oss-04' (storage incident)
RUN_ID_TO_TRIAGE = '8841'

**Run the agent.** Hand `run()` the model, the tools (wrapped in `StaticProvider`), the sandbox, the system prompt, and the task — then read back the answer, turn/cost stats, and the integrity check.

In [ ]:
result = await run(
    model=model,
    capabilities=StaticProvider(tools),
    sandbox=sandbox,
    system_prompt=system_prompt,
    task=f'Triage run {RUN_ID_TO_TRIAGE}. Use grep_log for big logs, read_log only when you need full context.',
    on_event=events.append,
)

print(result.content)
print()
print(f'turns={result.turns} tool_calls={result.tool_calls_made} cost=${result.cost_usd:.4f}')
print(f'chain valid: {result.verify_integrity().valid}')

## 5. Show the evidence

**Count the event types.** Every step the agent took is recorded; this tallies them by type.

In [ ]:
from collections import Counter

type_counts = Counter(e.type for e in events)
for t, n in type_counts.most_common():
    print(f'  {n:3d}  {t}')

**List the saved triage notes.** The durable artifact the agent produced — what an auditor would actually open.

In [ ]:
print('triage notes saved:')
for f in sorted(TRIAGE_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')

## Now build your own — guided form

Pick a real workflow from your lab. Fill in the five fields below
and the cell below them assembles a runnable agent. Reset, refine,
re-run.

Use these prompts:
- **Goal** — one sentence. *What does the agent finish doing?*
- **Knowledge** — paste a SKILL.md procedure for the domain.
- **Capabilities** — what tools does it need? (You'll wire the
  Python next.)
- **Boundaries** — what tools must it *never* call?
- **Evidence** — what would an auditor want to see in the trace?

**The five-question form.** Fill in Goal, Knowledge, Capabilities, Boundaries, and Evidence, then click *Render summary* to see your harness laid out. Edit and re-render as many times as you like.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

goal_in     = widgets.Text(value='', placeholder='e.g. Triage incoming SOC alerts against the playbook',
                           description='Goal:', layout=widgets.Layout(width='90%'), style={'description_width': 'initial'})
knowledge_in= widgets.Textarea(value=dedent('''
                  ---
                  name: my_skill
                  description: TODO
                  ---
                  # My Skill
                  
                  When given an alert:
                  1. TODO
                  2. TODO
                  3. TODO
              ''').strip(),
                  description='Knowledge (SKILL.md):', layout=widgets.Layout(width='90%', height='150px'),
                  style={'description_width': 'initial'})
caps_in     = widgets.Textarea(value='read_alert(alert_id)\nlookup_playbook(category)\nwrite_triage(alert_id, classification)',
                  description='Capabilities (one tool per line):', layout=widgets.Layout(width='90%', height='80px'),
                  style={'description_width': 'initial'})
bounds_in   = widgets.Textarea(value='No close_alert. No write to production. Output only to ../data/triage/.',
                  description='Boundaries:', layout=widgets.Layout(width='90%', height='60px'),
                  style={'description_width': 'initial'})
evidence_in = widgets.Textarea(value='Every alert id consulted, every playbook reference fetched, final classification, chain valid.',
                  description='Evidence:', layout=widgets.Layout(width='90%', height='60px'),
                  style={'description_width': 'initial'})
show_btn    = widgets.Button(description='Render summary ▶', button_style='primary')
out         = widgets.Output()

def _render(_):
    out.clear_output()
    with out:
        from rich.panel import Panel as _P; from rich.console import Console as _C
        c = _C()
        c.print(_P(f'GOAL:        {goal_in.value}\n'
                  f'KNOWLEDGE:\n{knowledge_in.value}\n\n'
                  f'CAPABILITIES:\n  ' + '\n  '.join(caps_in.value.splitlines()) + '\n\n'
                  f'BOUNDARIES:  {bounds_in.value}\n\n'
                  f'EVIDENCE:    {evidence_in.value}',
                  title='YOUR HARNESS — five questions answered', border_style='green'))

show_btn.on_click(_render)
display(widgets.VBox([goal_in, knowledge_in, caps_in, bounds_in, evidence_in, show_btn, out]))

## Wire your agent — runnable scaffold

✅ **TODO:** Implement the tools you listed in Capabilities. Set
`MY_GOAL` to your goal. Run the cell. The trace will tell you
whether the harness held.

**✅ Fill in your goal and skill.** Copy these from the form above — your one-sentence goal and your SKILL.md procedure.

In [ ]:
# === ✅ YOUR AGENT BELOW ===

MY_GOAL = 'TODO — copy from the form above'
MY_SKILL = dedent('''
    ---
    name: my_skill
    description: TODO
    ---
    # TODO — paste from the form above (knowledge field)
''').strip()

**✅ Implement at least one tool.** Replace the body of `my_tool` with a real capability from your list, then register it in `my_tools`.

In [ ]:
# ✅ TODO — implement at least one real tool from your Capabilities list.
async def my_tool(args, ctx: ToolContext) -> str:
    return 'TODO'

my_tools = [Tool(
    name='my_tool', description='TODO',
    input_schema={'type': 'object', 'properties': {}},
    execute=my_tool,
)]

**✅ Set your boundaries and task.** Allowlist only the tools that may fire, and give the agent something concrete to do.

In [ ]:
# ✅ TODO — boundaries. Only the tools listed here will fire.
my_sandbox = SandboxConfig(allowed_tools=['my_tool'])

# ✅ TODO — input that the goal applies to.
MY_TASK = 'TODO — give the agent something concrete to do.'

**✅ Run your agent.** Same call shape as the worked example. The trace tells you whether the harness held.

In [ ]:
events = []
result = await run(
    model=model,
    capabilities=StaticProvider(my_tools),
    sandbox=my_sandbox,
    system_prompt=f'{MY_GOAL}\n\n{MY_SKILL}',
    task=MY_TASK,
    on_event=events.append,
)
print(result.content)
print('chain valid:', result.verify_integrity().valid)
print('cost:', f'${result.cost_usd:.4f}')

## Closing

You started with a chat call. You ended with an audited,
sandboxed, skill-aware agent that runs a real workflow. The
total amount of *new* code you wrote was small.

What scaled was the **mental model**:

> *Goal · Knowledge · Capabilities · Boundaries · Evidence.*

That is portable. To LangChain. To LangGraph. To raw Python. To
whatever framework your lab adopts in 2027. The frameworks are
implementations of the model — the model is what you take with
you.